In [ ]:
# RF-DETR-Seg-Nano — Google Colab Training Pipeline

Dieses Notebook klont das Repository und führt die bestehenden Scripts der Reihe nach aus.

**Voraussetzung:** GPU-Laufzeit aktivieren: `Laufzeit → Laufzeittyp ändern → T4 GPU`

## 1. Repository klonen

In [ ]:
import os

REPO_URL = "https://github.com/Agredo/rf-detr-document.git"
REPO_DIR = "rf-detr-document"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL}")

os.chdir(REPO_DIR)
print(f"Arbeitsverzeichnis: {os.getcwd()}")
print("Repository-Inhalt:", os.listdir("."))

## 2. GPU prüfen

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "⚠️ Keine GPU gefunden – bitte GPU-Laufzeit aktivieren!")

## 3. Abhängigkeiten installieren

In [ ]:
%pip install -q -r requirements.txt
print("✓ Installation abgeschlossen.")

## 4. Datensatz hochladen

Navigiere im Datei-Dialog zum lokalen Dataset-Ordner und wähle **alle Dateien** auf einmal aus (Cmd+A):

```
/Users/agredo/Doument Detection/train/
```

Enthält: alle Bilddateien + `_annotations.coco.json` — einfach alle markieren und hochladen.

In [ ]:
import os, threading
from google.colab import files

UPLOAD_DIR = "roboflow_data/train"
os.makedirs(UPLOAD_DIR, exist_ok=True)

print("Alle Dateien aus dem Dataset-Ordner auswählen (Bilder + _annotations.coco.json):")
uploaded = files.upload()

# Parallel auf Disk schreiben
def save_file(name, data):
    path = os.path.join(UPLOAD_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    if name.endswith(".json"):
        print(f"  ✓ {name}")

threads = [threading.Thread(target=save_file, args=(n, d)) for n, d in uploaded.items()]
for t in threads: t.start()
for t in threads: t.join()

ROBOFLOW_DIR = UPLOAD_DIR
print(f"✓ {len(uploaded)} Dateien hochgeladen nach {ROBOFLOW_DIR}")
%env ROBOFLOW_DIR={ROBOFLOW_DIR}

## 5. Datensatz vorbereiten

Führt `scripts/prepare_dataset.py` aus — normalisiert Kategorien und erstellt den 80/20 Train/Valid-Split.

In [ ]:
import subprocess, os, sys

roboflow_dir = os.environ.get("ROBOFLOW_DIR", "roboflow_data")

result = subprocess.run(
    [sys.executable, "scripts/prepare_dataset.py",
     "--roboflow-dir", roboflow_dir,
     "--out-dir", "data",
     "--val-split", "0.2",
     "--seed", "42"],
    text=True
)
if result.returncode != 0:
    raise RuntimeError("prepare_dataset.py fehlgeschlagen")

In [ ]:
## 6. Training

Führt `scripts/train.py` aus. Parameter hier anpassen:

import subprocess, sys

# ── Trainingsparameter ──────────────────
RUN_NAME = "v1"
EPOCHS   = "75"
BATCH    = "8"
LR       = "1e-4"
WORKERS  = "2"
IMG_SIZE = "312"
# ────────────────────────────────────────

result = subprocess.run(
    [sys.executable, "scripts/train.py",
     "--run-name",    RUN_NAME,
     "--epochs",      EPOCHS,
     "--batch",       BATCH,
     "--lr",          LR,
     "--workers",     WORKERS,
     "--img-size",    IMG_SIZE],
    text=True
)
if result.returncode != 0:
    raise RuntimeError("train.py fehlgeschlagen")

In [ ]:
## 7. ONNX Export

Führt `scripts/export_onnx.py` aus.

import subprocess, sys

RUN_NAME = "v1"  # muss mit Training übereinstimmen

result = subprocess.run(
    [sys.executable, "scripts/export_onnx.py",
     "--run-name", RUN_NAME,
     "--device",   "cpu"],
    text=True
)
if result.returncode != 0:
    raise RuntimeError("export_onnx.py fehlgeschlagen")

In [ ]:
## 8. Ergebnisse herunterladen

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

RUN_NAME = "v1"
run_dir  = Path("runs") / RUN_NAME
zip_path = f"{RUN_NAME}_results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in run_dir.iterdir():
        z.write(f, f.name)

files.download(zip_path)
print(f"✓ {zip_path} wird heruntergeladen.")